## 1 · Setup & Data Loading

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", font_scale=1.05)
SPLIT_COLORS = {"train": "#2980B9", "devel": "#27AE60"}

DATA_DIR  = "preprocessed_data"
train_df  = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
devel_df  = pd.read_csv(os.path.join(DATA_DIR, "devel.csv"))
test_df   = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

RAW_FEATS = ["BPM", "ECG", "resp"]
LABEL_COL = "label"

print("Loaded successfully.")
print(f"  Train : {train_df.shape[0]:>6} rows, {train_df.shape[1]} columns")
print(f"  Devel : {devel_df.shape[0]:>6} rows, {devel_df.shape[1]} columns")
print(f"  Test  : {test_df.shape[0]:>6} rows, {test_df.shape[1]} columns")

## 2 · Dataset Overview

In [ ]:
print("Column names and dtypes (train):")
print(train_df.dtypes.to_string())


In [ ]:
def null_summary(df, name):
    s = df.isnull().sum()
    s = s[s > 0]
    if len(s) == 0:
        print(f"[{name}]  No missing values.")
    else:
        print(f"[{name}]  Missing values:")
        print(s.to_string())

for name, df in [("train", train_df), ("devel", devel_df), ("test", test_df)]:
    null_summary(df, name)


In [ ]:
train_df.head()

In [ ]:
train_df[RAW_FEATS].describe().round(4)

## 3 · Raw Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Raw Feature Distributions — Train Set", fontsize=13, fontweight="bold")

for i, feat in enumerate(RAW_FEATS):
    ax = axes[i]
    ax.hist(train_df[feat], bins=50, color="#5DADE2", edgecolor="white", alpha=0.85)
    ax.axvline(train_df[feat].mean(),   color="#E74C3C", lw=1.8, ls="--", label="mean")
    ax.axvline(train_df[feat].median(), color="#2ECC71", lw=1.8, ls=":",  label="median")
    ax.set_title(feat, fontsize=11, fontweight="bold")
    ax.set_xlabel("Value")
    ax.set_ylabel("Count" if i == 0 else "")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4 · Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Arousal Label Distribution — Train and Devel", fontsize=13, fontweight="bold")

for ax, (name, df) in zip(axes, [("Train", train_df), ("Devel", devel_df)]):
    col = df[LABEL_COL].dropna()
    ax.hist(col, bins=60, color=SPLIT_COLORS[name.lower()],
            edgecolor="white", alpha=0.8, density=True, label="histogram")
    kde_x = np.linspace(col.min(), col.max(), 400)
    ax.plot(kde_x, stats.gaussian_kde(col)(kde_x), color="#2C3E50", lw=2, label="KDE")
    ax.set_ylim(0)
    ax.set_title(f"{name}  (n = {len(col):,} rows)", fontsize=11)
    ax.set_xlabel("Arousal label"); ax.set_ylabel("Density")
    ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

## 5 · Correlation Analysis

In [ ]:
cols_for_corr = RAW_FEATS + [LABEL_COL]
corr = train_df[cols_for_corr].corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlBu_r",
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, linecolor="white", annot_kws={"size": 11})
ax.set_title("Pearson Correlation — BPM, ECG, resp, label (Train)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
feat_label_corr = train_df[RAW_FEATS + [LABEL_COL]].corr()[LABEL_COL].drop(LABEL_COL)

fig, ax = plt.subplots(figsize=(7, 3))
colours = ["#E74C3C" if v > 0 else "#5DADE2" for v in feat_label_corr.values]
ax.barh(feat_label_corr.index, feat_label_corr.values, color=colours, edgecolor="white")
ax.axvline(0, color="#2C3E50", lw=1.2)
ax.set_xlabel("Pearson r  with  label")
ax.set_title("Feature-Label Correlation (Train)", fontsize=11)
for i, v in enumerate(feat_label_corr.values):
    ax.text(v + (0.003 if v >= 0 else -0.003), i, f"{v:.3f}",
            va="center", ha="left" if v >= 0 else "right", fontsize=10)
plt.tight_layout()
plt.show()

## 6 · Subject-Level Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, df) in zip(axes, [("Train (41 subjects)", train_df), ("Devel (14 subjects)", devel_df)]):
    tps = df.groupby("subject_id").size().sort_values()
    colour = SPLIT_COLORS["train"] if "Train" in name else SPLIT_COLORS["devel"]
    ax.bar(tps.index.astype(str), tps.values, color=colour, edgecolor="white")
    ax.axhline(tps.mean(), color="#E74C3C", lw=1.8, ls="--", label=f"mean = {tps.mean():.0f}")
    ax.set_title(f"Timesteps per Subject — {name}", fontsize=11)
    ax.set_xlabel("Subject ID"); ax.set_ylabel("Timesteps")
    ax.tick_params(axis="x", rotation=60, labelsize=7)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
subj_stats = train_df.groupby("subject_id")[LABEL_COL].agg(["mean","std","min","max"]).round(4)
subj_stats = subj_stats.sort_values("mean")

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(subj_stats.index.astype(str), subj_stats["mean"], color="#5DADE2", edgecolor="white")
ax.errorbar(range(len(subj_stats)), subj_stats["mean"],
            yerr=subj_stats["std"], fmt="none", color="#2C3E50", lw=1.2, capsize=3)
ax.axhline(0, color="#95A5A6", lw=1, ls=":", label="zero")
ax.set_title("Subject-Level Mean Arousal Label +/- Std (Train, sorted by mean)", fontsize=11)
ax.set_xlabel("Subject ID (sorted by mean arousal)"); ax.set_ylabel("Arousal label")
ax.tick_params(axis="x", rotation=60, labelsize=8); ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
sample_subjects = [1, 8, 42]
fig, axes = plt.subplots(len(sample_subjects), 1, figsize=(13, 9), sharex=False)
fig.suptitle("Arousal Label Over Time — Sample Subjects (Train)", fontsize=12)

for ax, sid in zip(axes, sample_subjects):
    subj = train_df[train_df["subject_id"] == sid].copy().sort_values("timestamp")
    ax.plot(subj["timestamp"] / 1000, subj[LABEL_COL], color="#2980B9", lw=1.2, alpha=0.85)
    ax.axhline(0, color="#95A5A6", lw=1, ls=":", alpha=0.7)
    ax.set_ylabel("Arousal label")
    ax.set_title(f"Subject {sid}  ({len(subj)} timesteps)", fontsize=10)

axes[-1].set_xlabel("Time (seconds)")
plt.tight_layout()
plt.show()

## 7 · Train vs Devel Comparison

In [ ]:
print("Kolmogorov-Smirnov test — Train vs Devel (are distributions the same?)")
print(f"{'Feature':<22} {'KS stat':>8} {'p-value':>12} {'Conclusion':>22}")
print("-" * 68)
for col in RAW_FEATS:
    ks, pv = stats.ks_2samp(train_df[col].dropna(), devel_df[col].dropna())
    conclusion = "same distribution" if pv > 0.05 else "DIFFERENT (p<0.05)"
    print(f"{col:<22} {ks:>8.4f} {pv:>12.4f}   {conclusion}")


## 8 · Outlier Detection

In [ ]:
print("Outlier rows per feature (|z| > 3, computed against train stats) — Train set")
print(f"Total train rows: {len(train_df)}")
print()

train_mean = train_df[RAW_FEATS].mean()
train_std  = train_df[RAW_FEATS].std()
z_scores   = (train_df[RAW_FEATS] - train_mean) / train_std

print(f"{'Feature':<10} {'|z|>3 count':>12} {'%':>8}")
print("-" * 34)
for col in RAW_FEATS:
    n_out = (z_scores[col].abs() > 3).sum()
    print(f"{col:<10} {n_out:>12,} {n_out/len(train_df)*100:>7.2f}%")

any_outlier = (z_scores.abs() > 3).any(axis=1).sum()
print()
print(f"Rows with at least one |z|>3 feature: {any_outlier:,}  ({any_outlier/len(train_df)*100:.2f}%)")

## 9 · Key Findings Summary

| Finding | Detail |
|---------|--------|
| **Dataset size** | ~24,700 train / ~8,400 devel / ~8,400 test rows (one row per 500 ms timestep) |
| **Sampling rate** | 2 Hz — one sample every 500 ms |
| **Missing values** | None in features; label is NaN for test (withheld by challenge) |
| **Label range (train)** | Continuous arousal score, roughly −0.75 to +0.94 |
| **Task** | Regression — predict the continuous arousal score at each timestep |
| **BPM range** | ~60–160 bpm, mean ~99 bpm |
| **ECG range** | Near-zero mean (~−0.003 mV), small variance |
| **resp range** | High variance (std ~1.3); extreme values span −9.7 to +6.4 |
| **Feature-label correlation** | Weak for all features (|r| < 0.15) — arousal is multivariate |
| **Subject variability** | Large inter-subject arousal differences — baselines vary person to person |
| **Outliers (|z|>3)** | < 2% of rows per feature — no systematic noise detected |

In [ ]:
summary_rows = []
for name, df in [("Train", train_df), ("Devel", devel_df), ("Test", test_df)]:
    row = {
        "Split":             name,
        "Subjects":          df["subject_id"].nunique(),
        "Rows":              len(df),
        "Rows/Subj (mean)":  round(df.groupby("subject_id").size().mean(), 1),
    }
    if df[LABEL_COL].notna().any():
        row["label mean"] = round(df[LABEL_COL].mean(), 4)
        row["label std"]  = round(df[LABEL_COL].std(),  4)
        row["label min"]  = round(df[LABEL_COL].min(),  4)
        row["label max"]  = round(df[LABEL_COL].max(),  4)
    else:
        row["label mean"] = row["label std"] = row["label min"] = row["label max"] = "N/A"
    summary_rows.append(row)

pd.DataFrame(summary_rows).set_index("Split")